# 03 Sequence Statistics, Motifs, and k-mers

This notebook introduces sequence statistics and motif-based analysis for computational biology.

It supports **Module 04: Sequence Statistics, Motifs, and k-mers** and connects selected Rosalind problems such as `GC`, `SUBS`, `CONS`, `LCSM`, `KMER`, `PROB`, `EVAL`, and `RSTR`.

The goal is to show how DNA strings can be converted into interpretable summaries such as GC content, motif positions, consensus sequences, profile matrices, k-mer counts, and simple probability-based features.

## Learning Goals

After completing this notebook, a learner should be able to:

- compute GC content for DNA sequences;
- search for exact motifs in a sequence;
- build profile matrices from multiple DNA strings;
- compute a consensus sequence;
- find a shared motif using a simple brute-force approach;
- count k-mers;
- convert sequences into k-mer feature vectors;
- compute simple sequence probabilities using GC content;
- understand how sequence statistics become ML-ready features.

## Connection to Original Rosalind Solutions

This notebook is connected to my original Rosalind solutions preserved under:

- `original_rosalind_tracks/bioinformatics_stronghold/`
- `original_rosalind_tracks/bioinformatics_textbook_track/`

The notebook focuses on teaching-oriented versions of sequence statistics, motifs, k-mers, profile matrices, and probability-based sequence analysis. The original solution files remain the archive of solved Rosalind work, while this notebook explains and reorganizes the concepts for learning and future reuse.

## 1. Example DNA sequences

We will use a small collection of DNA sequences throughout this notebook.

In [ ]:
sequences = [
    "ATCCAGCT",
    "GGGCAACT",
    "ATGGATCT",
    "AAGCAACC",
    "TTGGAACT",
    "ATGCCATT",
    "ATGGCACT",
]

sequences

## 2. GC content

This connects to Rosalind problem `GC`.

GC content is the percentage of bases in a sequence that are either `G` or `C`.

In [ ]:
def gc_content(sequence: str) -> float:
    """Return GC content percentage for a DNA sequence."""
    if not sequence:
        return 0.0

    gc_count = sequence.count("G") + sequence.count("C")
    return (gc_count / len(sequence)) * 100


for sequence in sequences:
    print(sequence, gc_content(sequence))

## 3. Finding the sequence with highest GC content

This is a common FASTA-style task: compute GC content for every sequence and report the sequence with the highest value.

In [ ]:
def sequence_with_highest_gc(records: dict[str, str]) -> tuple[str, float]:
    """Return the record ID and GC content of the sequence with highest GC content."""
    best_id = None
    best_gc = -1.0

    for record_id, sequence in records.items():
        current_gc = gc_content(sequence)

        if current_gc > best_gc:
            best_id = record_id
            best_gc = current_gc

    return best_id, best_gc


records = {
    "Rosalind_1": "ATCCAGCT",
    "Rosalind_2": "GGGCAACT",
    "Rosalind_3": "ATGGATCT",
}

sequence_with_highest_gc(records)

## 4. Exact motif search

This connects to Rosalind problem `SUBS`.

A motif is a smaller sequence pattern that appears inside a larger sequence.

The function below returns one-based positions, matching Rosalind-style output.

In [ ]:
def find_motif_positions(sequence: str, motif: str) -> list[int]:
    """Return one-based positions where motif appears in sequence."""
    positions = []

    for i in range(len(sequence) - len(motif) + 1):
        if sequence[i:i + len(motif)] == motif:
            positions.append(i + 1)

    return positions


sequence = "GATATATGCATATACTT"
motif = "ATAT"

find_motif_positions(sequence, motif)

## 5. Profile matrix

This connects to Rosalind problem `CONS`.

A profile matrix counts how often each base appears at each position across a collection of equal-length DNA strings.

In [ ]:
def profile_matrix(sequences: list[str]) -> dict[str, list[int]]:
    """Return a profile matrix for equal-length DNA sequences."""
    if not sequences:
        return {"A": [], "C": [], "G": [], "T": []}

    sequence_length = len(sequences[0])
    profile = {
        "A": [0] * sequence_length,
        "C": [0] * sequence_length,
        "G": [0] * sequence_length,
        "T": [0] * sequence_length,
    }

    for sequence in sequences:
        for index, base in enumerate(sequence):
            profile[base][index] += 1

    return profile


profile = profile_matrix(sequences)
profile

## 6. Consensus sequence

The consensus sequence chooses the most frequent base at each position of the profile matrix.

In [ ]:
def consensus_sequence(profile: dict[str, list[int]]) -> str:
    """Return the consensus sequence from a profile matrix."""
    if not profile["A"]:
        return ""

    consensus = []

    for index in range(len(profile["A"])):
        best_base = max("ACGT", key=lambda base: profile[base][index])
        consensus.append(best_base)

    return "".join(consensus)


consensus = consensus_sequence(profile)

print(consensus)
print(profile)

## 7. Shared motif discovery

This connects to Rosalind problem `LCSM`.

The longest common shared motif is the longest substring that appears in every sequence.

The implementation below is simple and beginner-friendly. It is not optimized for very large datasets.

In [ ]:
def longest_shared_motif(sequences: list[str]) -> str:
    """Return the longest substring shared by all sequences."""
    if not sequences:
        return ""

    shortest_sequence = min(sequences, key=len)

    for length in range(len(shortest_sequence), 0, -1):
        for start in range(len(shortest_sequence) - length + 1):
            candidate = shortest_sequence[start:start + length]

            if all(candidate in sequence for sequence in sequences):
                return candidate

    return ""


shared_sequences = [
    "GATTACA",
    "TAGACCA",
    "ATACA",
]

longest_shared_motif(shared_sequences)

## 8. k-mer counting

This connects to Rosalind problem `KMER` and Textbook Track problems such as `BA1A` and `BA1B`.

A k-mer is a substring of length `k`. k-mer counts are one of the most common ways to represent biological sequences numerically.

In [ ]:
from collections import Counter


def kmer_counts(sequence: str, k: int) -> dict[str, int]:
    """Return counts of all observed k-mers in a sequence."""
    counts = Counter()

    for i in range(len(sequence) - k + 1):
        kmer = sequence[i:i + k]
        counts[kmer] += 1

    return dict(counts)


sequence = "ACGTTGCATGTCGCATGATGCATGAGAGCT"
kmer_counts(sequence, k=3)

## 9. Fixed-order k-mer feature vectors

For machine learning, it is often useful to convert sequences into fixed-length numerical vectors.

For DNA, there are `4^k` possible k-mers for a given `k`.

In [ ]:
from itertools import product


def all_dna_kmers(k: int) -> list[str]:
    """Return all possible DNA k-mers in lexicographic order."""
    return ["".join(kmer) for kmer in product("ACGT", repeat=k)]


def kmer_feature_vector(sequence: str, k: int) -> list[int]:
    """Return a fixed-order k-mer count vector for a DNA sequence."""
    observed_counts = kmer_counts(sequence, k)
    vocabulary = all_dna_kmers(k)

    return [observed_counts.get(kmer, 0) for kmer in vocabulary]


vocabulary = all_dna_kmers(2)
features = kmer_feature_vector("ACGTACGT", k=2)

print(vocabulary)
print(features)

## 10. Building a simple feature table

A collection of sequences can be converted into a table where each row is a sequence and each column is a feature.

This is an early bridge toward ML-ready bioinformatics.

In [ ]:
import pandas as pd


def kmer_feature_table(records: dict[str, str], k: int) -> pd.DataFrame:
    """Return a DataFrame of k-mer count features for multiple sequences."""
    vocabulary = all_dna_kmers(k)

    rows = []
    index = []

    for record_id, sequence in records.items():
        counts = kmer_counts(sequence, k)
        rows.append([counts.get(kmer, 0) for kmer in vocabulary])
        index.append(record_id)

    return pd.DataFrame(rows, index=index, columns=vocabulary)


feature_table = kmer_feature_table(records, k=2)
feature_table

## 11. Sequence probability from GC content

This connects to Rosalind problems such as `PROB`, `EVAL`, and `RSTR`.

If GC content is known, we can assign probabilities to each base:

- `P(G) = P(C) = GC / 2`
- `P(A) = P(T) = (1 - GC) / 2`

The probability of a sequence is the product of the probabilities of its bases.

In [ ]:
def sequence_probability_given_gc(sequence: str, gc_fraction: float) -> float:
    """Return probability of a DNA sequence under a GC-content model."""
    base_probability = {
        "G": gc_fraction / 2,
        "C": gc_fraction / 2,
        "A": (1 - gc_fraction) / 2,
        "T": (1 - gc_fraction) / 2,
    }

    probability = 1.0

    for base in sequence:
        probability *= base_probability[base]

    return probability


sequence_probability_given_gc("ACG", gc_fraction=0.5)

For long sequences, probabilities can become very small. Log probabilities are often more stable.

In [ ]:
import math


def log10_probability_given_gc(sequence: str, gc_fraction: float) -> float:
    """Return log10 probability of a DNA sequence under a GC-content model."""
    probability = sequence_probability_given_gc(sequence, gc_fraction)

    if probability == 0:
        return float("-inf")

    return math.log10(probability)


for gc_fraction in [0.25, 0.50, 0.75]:
    print(gc_fraction, log10_probability_given_gc("ACG", gc_fraction))

## 12. Expected motif occurrences

This connects to Rosalind problem `EVAL`.

If a motif has probability `p` at any given starting position and there are `n - k + 1` possible starting positions, then the expected number of occurrences is:

`(n - k + 1) * p`

In [ ]:
def expected_motif_occurrences(sequence_length: int, motif: str, gc_fraction: float) -> float:
    """Return expected number of motif occurrences under a GC-content model."""
    motif_probability = sequence_probability_given_gc(motif, gc_fraction)
    possible_positions = sequence_length - len(motif) + 1

    return possible_positions * motif_probability


expected_motif_occurrences(sequence_length=100, motif="ACG", gc_fraction=0.5)

## 13. Mini exercise set

Try modifying the functions above to solve these small exercises.

1. Write a function that returns AT content instead of GC content.
2. Modify `find_motif_positions()` so it returns zero-based Python indices.
3. Build a profile matrix for RNA sequences using `A`, `C`, `G`, and `U`.
4. Modify `longest_shared_motif()` to return all longest shared motifs if there is a tie.
5. Normalize the k-mer feature vector so values represent frequencies instead of counts.
6. Build a feature table that includes both GC content and k-mer features.
7. Compare the expected number of motif occurrences under multiple GC-content values.

## Summary

This notebook showed how biological sequences can be summarized using statistics, motifs, profiles, k-mers, and simple probability models.

| Concept | Bioinformatics Use |
|---|---|
| GC content | sequence composition |
| Motif search | finding biologically meaningful patterns |
| Profile matrix | summarizing multiple aligned sequences |
| Consensus sequence | representative sequence construction |
| Shared motif | conserved sequence discovery |
| k-mer counts | feature extraction and sequence representation |
| Sequence probability | probabilistic modelling |
| Expected motif occurrence | statistical interpretation of motif search |

These ideas prepare the learner for later modules on string matching, genome indexing, practical Biopython workflows, and ML-ready sequence feature construction.